In [ ]:
!pip install transformers seqeval evaluate accelerate -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=6acd2128864dd445ecb9361752be5510cc1dd649dfbb787f5fb05c21e2d6609f
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import evaluate
seqeval = evaluate.load("seqeval") # Khai báo seqeval ở đây
def compute_metrics(p):
    predictions, labels = p

    # Biến ma trận xác suất (logits) thành con số ID nhãn dự đoán cao nhất
    predictions = np.argmax(predictions, axis=2)

    # Lọc bỏ các nhãn -100 ra khỏi quá trình tính toán metric
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
def load_conll_data(file_path):
    sentences = []
    current_sentence = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    word, tag = parts[0], parts[1]
                    current_sentence.append((word, tag))
        if current_sentence:
            sentences.append(current_sentence)
    return sentences

# Đọc tập train, dev
train_sentences = load_conll_data('/content/drive/MyDrive/datasetViMedNER/traindata/train.txt')
dev_sentences = load_conll_data('/content/drive/MyDrive/datasetViMedNER/traindata/dev.txt')

In [ ]:
# Đọc danh sách nhãn từ file labels.txt có sẵn trong thư mục train_dataset
label_path = '/content/drive/MyDrive/datasetViMedNER/traindata/labels.txt'

with open(label_path, 'r', encoding='utf-8') as f:
    unique_tags = [line.strip() for line in f if line.strip()]

# Tạo lại từ điển ánh xạ
label2id = {tag: idx for idx, tag in enumerate(unique_tags)}
id2label = {idx: tag for idx, tag in enumerate(unique_tags)}

print(f"✅ Đã tải thành công {len(label2id)} nhãn từ file labels.txt!")

✅ Đã tải thành công 11 nhãn từ file labels.txt!


In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

class ViMedNERDataset(Dataset):
    def __init__(self, sentences, tokenizer, label2id, max_length=128):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence_data = self.sentences[idx]
        words = [item[0] for item in sentence_data]
        tags = [item[1] for item in sentence_data]

        # Tokenize từng từ và theo dõi số lượng sub-token của mỗi từ
        tokenized_outputs = []
        label_ids = []

        # Thêm token [CLS] đầu câu
        tokenized_outputs.append(self.tokenizer.cls_token_id)
        label_ids.append(-100)

        for word, tag in zip(words, tags):
            # Tokenize từng từ lẻ (không dùng is_split_into_words để tránh lỗi word_ids)
            sub_tokens = self.tokenizer.tokenize(word)
            sub_token_ids = self.tokenizer.convert_tokens_to_ids(sub_tokens)

            if len(sub_token_ids) > 0:
                # Sub-token đầu tiên nhận nhãn thật
                tokenized_outputs.append(sub_token_ids[0])
                label_ids.append(self.label2id[tag])

                # Các sub-token phía sau của cùng 1 từ nhận -100
                for sub_id in sub_token_ids[1:]:
                    tokenized_outputs.append(sub_id)
                    label_ids.append(-100)

        # Thêm token [SEP] cuối câu
        tokenized_outputs.append(self.tokenizer.sep_token_id)
        label_ids.append(-100)

        # Cắt ngắn (Truncation) nếu vượt quá max_length
        if len(tokenized_outputs) > self.max_length:
            tokenized_outputs = tokenized_outputs[:self.max_length]
            label_ids = label_ids[:self.max_length]

        # Tạo attention mask (1 cho token thật, 0 cho padding)
        attention_mask = [1] * len(tokenized_outputs)

        # Padding (Đệm) cho đủ max_length
        padding_length = self.max_length - len(tokenized_outputs)
        if padding_length > 0:
            tokenized_outputs = tokenized_outputs + [self.tokenizer.pad_token_id] * padding_length
            label_ids = label_ids + [-100] * padding_length
            attention_mask = attention_mask + [0] * padding_length

        item = {
            "input_ids": torch.tensor(tokenized_outputs, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long)
        }
        return item

# 1. Khởi tạo Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

# 2. Khởi tạo lại Dataset
train_dataset = ViMedNERDataset(train_sentences, tokenizer, label2id)
dev_dataset = ViMedNERDataset(dev_sentences, tokenizer, label2id)

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

- gán nhãn -100 cho các sub_token đằng sau phần được tách ra đầu tiên của từ có nhãn, khi huấn luyện model tự biến để bỏ qua khi tính loss tránh học 2 sub_token của cùng 1 từ cần nhận biết
- attention mask sẽ giúp tạo ra những giá trị 0 cho seft-attention khi tính để bỏ qua attention của pad, cls, sep

In [ ]:
# Xem thử 2 mẫu đầu tiên trong train_dataset
for i in range(2):
    sample = train_dataset[i]
    print(f"--- Mẫu số {i} ---")
    for key, val in sample.items():
        print(f"{key}: {val.shape}")

--- Mẫu số 0 ---
input_ids: torch.Size([128])
attention_mask: torch.Size([128])
labels: torch.Size([128])
--- Mẫu số 1 ---
input_ids: torch.Size([128])
attention_mask: torch.Size([128])
labels: torch.Size([128])


In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='seqeval')
# 1. Khởi tạo mô hình PhoBERT
model = AutoModelForTokenClassification.from_pretrained(
    "vinai/phobert-base-v2",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# 2. Tạo DataCollator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 3. Cấu hình tham số huấn luyện
training_args = TrainingArguments(
    output_dir="./vimedner_baseline",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

# 4. Khởi tạo Trainer (đã lược bỏ dòng truyền tokenizer vào đây)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# 5. Tiến hành huấn luyện Baseline
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.560534,0.464422,0.586962,0.611544,0.599001,0.883114
2,0.413455,0.393071,0.620482,0.663624,0.641328,0.894919
3,0.352564,0.363878,0.622618,0.684295,0.652002,0.896468
4,0.293446,0.365890,0.588515,0.698792,0.638930,0.889649
5,0.260711,0.363578,0.609321,0.705503,0.653894,0.894117


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1430, training_loss=0.4194902006562773, metrics={'train_runtime': 677.3895, 'train_samples_per_second': 33.755, 'train_steps_per_second': 2.111, 'total_flos': 1493759120666880.0, 'train_loss': 0.4194902006562773, 'epoch': 5.0})

In [ ]:
from transformers import pipeline

# Tạo ống kính (pipeline) nhận mô hình tốt nhất của bạn
nlp_ner = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    device=model.device
)

# Test thử luôn với câu của bạn
text = "Bệnh nhân có tiền sử cao huyết áp và đái tháo đường tuýp 2 đang điều trị sau đó không chữa được nên dẫn đến run tay chân, tuyến tiền liệt bị chảy máu."
results = nlp_ner(text)

for item in results:
    print(f"{item['word']}: {item['entity']}")

cao: B-ten_benh
huyết: I-ten_benh
áp: I-ten_benh
đái: B-ten_benh
tháo: I-ten_benh
đường: I-ten_benh
tuýp: I-ten_benh
2: I-ten_benh
run: B-trieu_chung_benh
tay: I-trieu_chung_benh
châ@@: I-trieu_chung_benh
chảy: B-trieu_chung_benh
má@@: I-trieu_chung_benh


In [ ]:
import numpy as np
import pandas as pd
import evaluate
from sklearn.metrics import classification_report as sklearn_report
from IPython.display import display

seqeval = evaluate.load("seqeval") # Khai báo seqeval ở đây
def compute_metrics(p):
    predictions, labels = p

    # Biến ma trận xác suất (logits) thành con số ID nhãn dự đoán cao nhất
    predictions = np.argmax(predictions, axis=2)

    # Lọc bỏ các nhãn -100 ra khỏi quá trình tính toán metric
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

def compute_eval_classify_metrics(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)

    # 1. Lọc nhãn -100 và giữ nguyên cấu trúc List of Lists (dành cho Seqeval)
    true_predictions_seq = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels_seq = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # 2. Trải phẳng thành mảng 1 chiều (dành cho Sklearn)
    true_predictions_flat = [tag for sent in true_predictions_seq for tag in sent]
    true_labels_flat = [tag for sent in true_labels_seq for tag in sent]

    metrics_dict = {}

    # =========================================================
    # BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ HOÀN CHỈNH (SEQEVAL)
    # =========================================================
    results_seq = seqeval.compute(predictions=true_predictions_seq, references=true_labels_seq)
    table_entity = []

    for key, value in results_seq.items():
        if isinstance(value, dict):
            table_entity.append({
                "Thực thể (Entity)": key,
                "Precision": f"{value['precision']:.4f}",
                "Recall": f"{value['recall']:.4f}",
                "F1-Score": f"{value['f1']:.4f}",
                "Number (Entities)": value['number']
            })
            metrics_dict[f"entity_{key}_f1"] = value["f1"]

    table_entity.append({
        "Thực thể (Entity)": "OVERALL",
        "Precision": f"{results_seq['overall_precision']:.4f}",
        "Recall": f"{results_seq['overall_recall']:.4f}",
        "F1-Score": f"{results_seq['overall_f1']:.4f}",
        "Number (Entities)": "-"
    })
    metrics_dict["overall_f1"] = results_seq['overall_f1']

    print("\n" + "="*75)
    print("📊 BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ (STRICT ENTITY LEVEL)")
    print("="*75)
    display(pd.DataFrame(table_entity))

    # =========================================================
    # BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN RỜI RẠC (SKLEARN)
    # =========================================================
    report_tag = sklearn_report(true_labels_flat, true_predictions_flat, output_dict=True, zero_division=0)
    table_tag = []

    for key, value in report_tag.items():
        if key in ["accuracy", "macro avg", "weighted avg"]:
            continue
        table_tag.append({
            "Nhãn (Tag)": key,
            "Precision": f"{value['precision']:.4f}",
            "Recall": f"{value['recall']:.4f}",
            "F1-Score": f"{value['f1-score']:.4f}",
            "Number (Tokens)": int(value['support'])
        })

    overall_tag = report_tag["weighted avg"]
    table_tag.append({
        "Nhãn (Tag)": "OVERALL (Weighted)",
        "Precision": f"{overall_tag['precision']:.4f}",
        "Recall": f"{overall_tag['recall']:.4f}",
        "F1-Score": f"{overall_tag['f1-score']:.4f}",
        "Number (Tokens)": int(overall_tag['support'])
    })

    print("\n" + "="*75)
    print("📊 BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN (TOKEN TAG LEVEL: B-, I-, O)")
    print("="*75)
    display(pd.DataFrame(table_tag))

    return metrics_dict

In [ ]:
import pickle
def load_dataset(file_path):
    with open(file_path,'rb') as f :
        dataset = pickle.load(f)
    return dataset

dev_dataset_filepath = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/dev_dataset.pkl'
dev_dataset = load_dataset(dev_dataset_filepath)

In [ ]:
import os
import torch
import pandas as pd
import safetensors.torch
from transformers import Trainer, TrainingArguments
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, AutoTokenizer
from IPython.display import display

data_collator = DataCollatorForTokenClassification(tokenizer)

# 1. Khởi tạo danh sách lưu kết quả
results_table = []

# Cấu hình Trainer cơ bản chỉ dùng để đánh giá (Inference)
eval_args = TrainingArguments(
    output_dir="./eval_temp",
    per_device_eval_batch_size=16,
    report_to="none"
)

# 2. Hàm helper tự động nạp trọng số và lấy điểm
def trainer_model(model_name, model_instance, checkpoint_path, dataset):
    print(f"⏳ Đang tải và đánh giá: {model_name}...")

    bin_path = os.path.join(checkpoint_path, "pytorch_model.bin")
    safetensor_path = os.path.join(checkpoint_path, "model.safetensors")

    # 1. Ưu tiên tuyệt đối nạp file pytorch_model.bin cho các mô hình Custom
    if os.path.exists(bin_path):
        state_dict = torch.load(bin_path, map_location="cpu")
        model_instance.load_state_dict(state_dict, strict=False)
        print(f"✅ Đã nạp thành công trọng số từ: pytorch_model.bin")

    # 2. Nếu không có file bin (ví dụ Baseline), mới dùng safetensors
    elif os.path.exists(safetensor_path):
        import safetensors.torch
        safetensors.torch.load_model(model_instance, safetensor_path)
        print(f"✅ Đã nạp thành công trọng số từ: model.safetensors")

    else:
        print(f"⚠️ Cảnh báo: Không tìm thấy file trọng số tại {checkpoint_path}")
        return None

    model_instance.eval()

    # Tạo Trainer để chạy dự đoán
    eval_trainer = Trainer(
        model=model_instance,
        args=eval_args,
        eval_dataset=dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )
    return eval_trainer

In [ ]:
num_labels = len(label2id)
# baseline_trainer
model_baseline = AutoModelForTokenClassification.from_pretrained("vinai/phobert-base-v2", num_labels=num_labels)
trainer_baseline = trainer_model("1. Baseline (CE Loss)", model_baseline, "/content/drive/MyDrive/vimedner_final_baseline", dev_dataset)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


⏳ Đang tải và đánh giá: 1. Baseline (CE Loss)...
✅ Đã nạp thành công trọng số từ: model.safetensors


In [ ]:
output = trainer_baseline.predict(dev_dataset)

# Chỉ truyền vào tuple gồm 2 giá trị là predictions và label_ids
compute_eval_classify_metrics((output.predictions, output.label_ids))


📊 BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ (STRICT ENTITY LEVEL)


,Thực thể (Entity),Precision,Recall,F1-Score,Number (Entities)
0,bien_phap_chan_doan,0.5506,0.7232,0.6252,271
1,bien_phap_dieu_tri,0.5590,0.7473,0.6396,653
2,nguyen_nhan_benh,0.1653,0.1544,0.1597,259
3,ten_benh,0.7792,0.8747,0.8241,1795
4,trieu_chung_benh,0.5780,0.7095,0.6370,747
5,OVERALL,0.6414,0.7581,0.6949,-



📊 BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN (TOKEN TAG LEVEL: B-, I-, O)


,Nhãn (Tag),Precision,Recall,F1-Score,Number (Tokens)
0,B-bien_phap_chan_doan,0.7248,0.7970,0.7592,271
1,B-bien_phap_dieu_tri,0.7139,0.8407,0.7722,653
2,B-nguyen_nhan_benh,0.7941,0.1042,0.1843,259
3,B-ten_benh,0.8448,0.9281,0.8845,1795
4,B-trieu_chung_benh,0.7150,0.7791,0.7457,747
5,I-bien_phap_chan_doan,0.7472,0.7813,0.7639,942
6,I-bien_phap_dieu_tri,0.7286,0.7415,0.7350,1745
7,I-nguyen_nhan_benh,0.5098,0.4894,0.4994,850
8,I-ten_benh,0.8845,0.9516,0.9168,4585
9,I-trieu_chung_benh,0.7206,0.7694,0.7442,1522


{'entity_bien_phap_chan_doan_f1': np.float64(0.6251993620414673),
 'entity_bien_phap_dieu_tri_f1': np.float64(0.6395806028833552),
 'entity_nguyen_nhan_benh_f1': np.float64(0.1596806387225549),
 'entity_ten_benh_f1': np.float64(0.8241469816272965),
 'entity_trieu_chung_benh_f1': np.float64(0.6370192307692308),
 'overall_f1': np.float64(0.6948818897637795)}